# Multi-Armed Bandit Problem

### Bandit Env

In [3]:
import numpy as np
import matplotlib.pyplot as plt


class BanditEnv:

    def __init__(self, true_means, reward_std=1.0, seed=None):
        self.true_means = np.array(true_means, dtype=float)
        self.k          = len(self.true_means)
        self.reward_std = reward_std
        self.rng        = np.random.default_rng(seed)

    def pull(self, action):
        """Pull arm `action` and return a noisy reward sample."""
        return float(self.rng.normal(self.true_means[action], self.reward_std))

### The Agent

In [1]:
class Agent:

    def __init__(self, k, epsilon=0.1, seed=None):
        self.k       = k
        self.epsilon = epsilon
        self.rng     = np.random.default_rng(seed)

        self.Q = np.zeros(k)       # Q(a): estimated value for each arm
        self.N = np.zeros(k, dtype=int)  # N(a): how many times each arm was pulled

    def select_action(self):
        """Choose an arm using the ε-greedy policy."""
        if self.rng.random() < self.epsilon: # %10
            # Explore: pick any arm at random
            return int(self.rng.integers(self.k))
        else:
            # Exploit: pick the arm with the highest current estimate
            # If multiple arms are tied, break the tie randomly
            max_q     = np.max(self.Q)
            best_arms = np.flatnonzero(self.Q == max_q)
            return int(self.rng.choice(best_arms))

    def update(self, action, reward):
        """
        Update the Q-value estimate for `action` using the
        incremental sample-average formula.
        """
        self.N[action] += 1
        # Move Q(a) a step of size 1/N towards the new reward
        self.Q[action] += (reward - self.Q[action]) / self.N[action]

### Run an Episode

At each step:
1. The agent picks an arm.
2. The environment returns a reward.
3. The agent updates its Q-value estimate.

In [14]:
true_means = [1.0, 1.1, 1.2]
seed = 39
env = BanditEnv(true_means, reward_std=10.0, seed=seed)

epsilon = 0.1
agent = Agent(k=len(true_means), epsilon=epsilon, seed=seed)

episodes = 1000

rewards = np.zeros(episodes)
actions = np.zeros(episodes, dtype=int)

for i in range(episodes):
    action = agent.select_action()
    reward = env.pull(action)
    agent.update(action, reward)

    rewards[i] = reward
    actions[i] = action


In [15]:
print("True means:          ", true_means)
print()
print("Learned Q-values:    ", np.round(agent.Q, 3))
print("Pull counts N(a):    ", agent.N)
print()
print(f"Total reward:        {rewards.sum():.1f}")
print(f"Average reward:      {rewards.mean():.3f}")
best_arm = int(np.argmax(env.true_means))
print(f"Optimal arm chosen:  {(actions == best_arm).mean() * 100:.1f}% of the time")

True means:           [1.0, 1.1, 1.2]

Learned Q-values:     [1.153 0.577 1.571]
Pull counts N(a):     [ 52  65 883]

Total reward:        1484.2
Average reward:      1.484
Optimal arm chosen:  88.3% of the time
